# 📓 Semana 20 · Dia 3 — Editor YAML admin com diff e rollback

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (app Streamlit) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | Portfólio empresarial |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Editor com diff/rollback funcional |

---


## 📖 Teoria — A seção admin

O admin gerencia os fluxos: cria/edita YAML, vê o **diff** da alteração e faz **rollback** para uma versão anterior. Cada salvar vira uma **versão** do fluxo (versionado em tabela).


### 💻 Na prática — Versionando fluxos

Toda alteração salva uma nova versão.


In [ ]:
# Salvar versão do fluxo
from pyspark.sql.functions import current_timestamp
spark.sql("CREATE TABLE IF NOT EXISTS workspace.app.fluxo_versoes (
  fluxo_id STRING, versao INT, yaml_def STRING, editado_por STRING,
  editado_em TIMESTAMP) USING DELTA")
def salvar_versao(fluxo_id, yaml_def, usuario):
    v = spark.sql(f"SELECT COALESCE(MAX(versao),0)+1 AS v FROM workspace.app.fluxo_versoes WHERE fluxo_id = '{fluxo_id}'").collect()[0][0]
    spark.createDataFrame([(fluxo_id, v, yaml_def, usuario, "now")],
                          ["fluxo_id", "versao", "yaml_def", "editado_por", "editado_em"])\
        .withColumn("editado_em", current_timestamp())\
        .write.mode("append").saveAsTable("workspace.app.fluxo_versoes")
    return v
print("salvar_versao pronto — cada edição vira versão.")

In [ ]:
# Diff entre versões (didático)
import difflib
def diff_yaml(v_antiga, v_nova):
    return "\n".join(difflib.unified_diff(
        v_antiga.splitlines(), v_nova.splitlines(), lineterm=""))
print(diff_yaml("fluxo: metas\nnome: Metas", "fluxo: metas\nnome: Metas 2026"))

### 💻 Na prática — Rollback

Restaurar uma versão anterior = salvar o YAML antigo como nova versão (nunca apagar o histórico).


In [ ]:
# Rollback (nova versão com o YAML antigo)
def rollback(fluxo_id, versao_alvo, usuario):
    yaml_antigo = spark.sql(f"SELECT yaml_def FROM workspace.app.fluxo_versoes WHERE fluxo_id='{fluxo_id}' AND versao={versao_alvo}").collect()[0][0]
    salvar_versao(fluxo_id, yaml_antigo, usuario)
    print(f"Rollback para v{versao_alvo} feito (nova versão criada).")
print("Rollback sem perder histórico — auditoria completa.")

> 🎯 **Dica de prova**: Portfólio: versionar configuração (YAML) com diff/rollback é o padrão de produto admin — mesmos princípios de Git aplicados a configuração de negócio.


## 🎯 Exercícios de fixação

**1.** Adicione a listagem de versões no app admin.

**2.** Por que rollback cria versão nova em vez de apagar?

**3.** Monte o editor com text_area + botão salvar (Streamlit).


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Listagem

SELECT versao, editado_em, editado_por FROM fluxo_versoes ORDER BY versao DESC.

**2.** Nova versão

Histórico íntegro e auditável — apagar versão destruiria a trilha de auditoria.

**3.** Editor

st.text_area (YAML) → validar (JSON Schema) → salvar_versao → mostrar diff.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*